In [9]:
import pandas as pd
import statsmodels.api as sm

# (1) 데이터 불러오기
df_y = pd.read_csv('Y_it_continued_claims_panel.csv', parse_dates=['Filed week ended'])
df_x = pd.read_csv('X_it_for_regression.csv')
df_doge = pd.read_csv('doge_intensity_weighted_average_full.csv')

# (2) 컬럼 이름 정리
df_doge = df_doge.rename(columns={'state_name': 'State'})

# (3) 데이터 병합
df_merged = df_y.merge(df_doge, on='State', how='left')
df_merged = df_merged.merge(df_x, on='State', how='left')

# (4) 2022~2025년 데이터만 필터링
df_merged = df_merged[(df_merged['Filed week ended'] >= '2022-01-01') & (df_merged['Filed week ended'] <= '2025-12-31')]

# (5) POST 더미 변수 생성
df_merged['POST'] = (df_merged['Filed week ended'] >= pd.to_datetime('2025-01-01')).astype(int)

# (6) DOGE_Intensity × POST 상호작용항 생성
df_merged['DOGE_Intensity_POST'] = df_merged['DOGE_Intensity_EmploymentBased'] * df_merged['POST']

# (7) 독립변수 및 종속변수 설정
from sklearn.decomposition import PCA

# 산업구조 변수만 추출
industry_cols = [col for col in df_x.columns if col not in ['State', 'Population_Density', 'College_Educated_Share']]
X_industry = df_merged[industry_cols]

# 결측치 처리 (필요하면)
X_industry = X_industry.fillna(0)

# PCA 2개 성분으로 축소
pca = PCA(n_components=2)
industry_pca = pca.fit_transform(X_industry)

# DataFrame으로 합치기
df_merged['Industry_PC1'] = industry_pca[:, 0]
df_merged['Industry_PC2'] = industry_pca[:, 1]

# 회귀 독립변수 설정
indep_vars = [
    'POST',
    'DOGE_Intensity_POST',
    'Population_Density',
    'College_Educated_Share',
    'Industry_PC1',
    'Industry_PC2'
]


X = df_merged[indep_vars]
X = sm.add_constant(X)
y = df_merged['Continued Claims']
groups = df_merged['State']

# (8) missing='drop' 처리
valid_idx = X.dropna().index.intersection(y.dropna().index)
X = X.loc[valid_idx]
y = y.loc[valid_idx]
groups = groups.loc[valid_idx]


# (9) 회귀 실행 (State별 클러스터 표준오차)
model = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': groups})

# 결과 출력
model.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:       Continued Claims   R-squared:                       0.173
Model:                            OLS   Adj. R-squared:                  0.172
Method:                 Least Squares   F-statistic:                     4.652
Date:                Sat, 10 May 2025   Prob (F-statistic):           0.000790
Time:                        21:00:35   Log-Likelihood:            -1.0806e+05
No. Observations:                8772   AIC:                         2.161e+05
Df Residuals:                    8765   BIC:                         2.162e+05
Df Model:                           6                                         
Covariance Type:              cluster                                         
==========================================================================================
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                   1.074e+05   4.52e+04      2.376      0.018    1.88e+04    1.96e+05
POST                    2.745e+04   1.13e+04      2.422      0.015    5235.010    4.97e+04
DOGE_Intensity_POST    -6.452e+06   4.19e+06     -1.541      0.123   -1.47e+07    1.75e+06
Population_Density        -8.9504      5.929     -1.510      0.131     -20.572       2.671
College_Educated_Share  -1.53e+05   9.71e+04     -1.576      0.115   -3.43e+05    3.73e+04
Industry_PC1           -2.962e+06   1.41e+06     -2.100      0.036   -5.73e+06   -1.98e+05
Industry_PC2            6.295e+05   4.15e+05      1.519      0.129   -1.83e+05    1.44e+06
==============================================================================
Omnibus:                     6878.344   Durbin-Watson:                   0.021
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           141990.984
Skew:                           3.710   Prob(JB):                         0.00
Kurtosis:                      21.260   Cond. No.                     2.15e+06
==============================================================================

Notes:
[1] Standard Errors are robust to cluster correlation (cluster)
[2] The condition number is large, 2.15e+06. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [10]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# (1) PCA로 Industry_PC1, Industry_PC2 만들기
industry_cols = [col for col in df_x.columns if col not in ['State', 'Population_Density', 'College_Educated_Share']]
X_industry = df_merged[industry_cols].fillna(0)

pca = PCA(n_components=2)
industry_pca = pca.fit_transform(X_industry)

df_merged['Industry_PC1'] = industry_pca[:, 0]
df_merged['Industry_PC2'] = industry_pca[:, 1]

# (2) 스케일링
to_scale = ['Population_Density', 'College_Educated_Share', 'Industry_PC1', 'Industry_PC2']
scaler = StandardScaler()
df_merged_scaled = df_merged.copy()
df_merged_scaled[to_scale] = scaler.fit_transform(df_merged_scaled[to_scale])

# (3) 독립변수 설정
indep_vars = [
    'POST',
    'DOGE_Intensity_POST',
    'Population_Density',
    'College_Educated_Share',
    'Industry_PC1',
    'Industry_PC2'
]

X = df_merged_scaled[indep_vars]
X = sm.add_constant(X)
y = df_merged_scaled['Continued Claims']
groups = df_merged_scaled['State']

# (4) 결측치 처리
valid_idx = X.dropna().index.intersection(y.dropna().index)
X = X.loc[valid_idx]
y = y.loc[valid_idx]
groups = groups.loc[valid_idx]

# (5) 회귀 실행
model = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': groups})

# 결과 출력
model.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:       Continued Claims   R-squared:                       0.173
Model:                            OLS   Adj. R-squared:                  0.172
Method:                 Least Squares   F-statistic:                     4.652
Date:                Sat, 10 May 2025   Prob (F-statistic):           0.000790
Time:                        21:01:06   Log-Likelihood:            -1.0806e+05
No. Observations:                8772   AIC:                         2.161e+05
Df Residuals:                    8765   BIC:                         2.162e+05
Df Model:                           6                                         
Covariance Type:              cluster                                         
==========================================================================================
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                   4.993e+04   1.46e+04      3.428      0.001    2.14e+04    7.85e+04
POST                    2.745e+04   1.13e+04      2.422      0.015    5235.010    4.97e+04
DOGE_Intensity_POST    -6.452e+06   4.19e+06     -1.541      0.123   -1.47e+07    1.75e+06
Population_Density     -1.234e+04   8177.016     -1.510      0.131   -2.84e+04    3683.247
College_Educated_Share -1.088e+04   6907.076     -1.576      0.115   -2.44e+04    2653.939
Industry_PC1           -8.933e+04   4.25e+04     -2.100      0.036   -1.73e+05   -5960.050
Industry_PC2            1.497e+04   9857.908      1.519      0.129   -4349.714    3.43e+04
==============================================================================
Omnibus:                     6878.344   Durbin-Watson:                   0.021
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           141990.984
Skew:                           3.710   Prob(JB):                         0.00
Kurtosis:                      21.260   Cond. No.                     2.19e+03
==============================================================================

Notes:
[1] Standard Errors are robust to cluster correlation (cluster)
[2] The condition number is large, 2.19e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [11]:
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from statsmodels.stats.outliers_influence import variance_inflation_factor

# (1) 데이터 불러오기
df_y = pd.read_csv('Y_it_continued_claims_panel.csv', parse_dates=['Filed week ended'])
df_x = pd.read_csv('X_it_for_regression.csv')
df_doge = pd.read_csv('doge_intensity_weighted_average_full.csv')
df_doge = df_doge.rename(columns={'state_name': 'State'})

# (2) DOGE Intensity 상위 10개 주 추출
top10_states = df_doge[['State', 'DOGE_Intensity_WeightedAverage']].sort_values(
    by='DOGE_Intensity_WeightedAverage', ascending=False).head(10)['State'].tolist()

# (3) 데이터 병합
df_merged = df_y.merge(df_doge, on='State', how='left')
df_merged = df_merged.merge(df_x, on='State', how='left')

# (4) 필터링 및 POST 생성
df_merged = df_merged[(df_merged['Filed week ended'] >= '2022-01-01') & (df_merged['Filed week ended'] <= '2025-12-31')]
df_merged['POST'] = (df_merged['Filed week ended'] >= pd.to_datetime('2025-01-01')).astype(int)

# (5) 산업구조 PCA
industry_cols = [col for col in df_x.columns if col not in ['State', 'Population_Density', 'College_Educated_Share']]
X_industry = df_merged[industry_cols].fillna(0)
pca = PCA(n_components=2)
industry_pca = pca.fit_transform(X_industry)
df_merged['Industry_PC1'] = industry_pca[:, 0]
df_merged['Industry_PC2'] = industry_pca[:, 1]

# (6) 상호작용항 생성 및 중심화
df_merged['DOGE_Intensity_c'] = df_merged['DOGE_Intensity_WeightedAverage'] - df_merged['DOGE_Intensity_WeightedAverage'].mean()
df_merged['POST_c'] = df_merged['POST'] - df_merged['POST'].mean()
df_merged['DOGE_Intensity_POST_c'] = df_merged['DOGE_Intensity_c'] * df_merged['POST_c']

# (7) 상위 10개 주 필터링
df_top10 = df_merged[df_merged['State'].isin(top10_states)]

# (8) 표준화
scaler = StandardScaler()
to_scale = ['Population_Density', 'College_Educated_Share', 'Industry_PC1', 'Industry_PC2', 'DOGE_Intensity_POST_c']
df_top10[to_scale] = scaler.fit_transform(df_top10[to_scale])

# (9) 독립변수 설정
indep_vars = [
    'POST_c',
    'DOGE_Intensity_POST_c',
    'Population_Density',
    'College_Educated_Share',
    'Industry_PC1',
    'Industry_PC2'
]
X = df_top10[indep_vars]
X = sm.add_constant(X)
y = df_top10['Continued Claims']
groups = df_top10['State']

# (10) NA 제거
full_data = pd.concat([X, y, groups], axis=1).dropna()
X = full_data[X.columns]
y = full_data['Continued Claims']
groups = full_data['State']

# (11) VIF 계산
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("\nVIF 점검 결과:")
print(vif_data)

# (12) 개선된 OLS 회귀
model_top10 = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': groups})

# (13) 결과 출력
print("\nOLS 회귀 결과:")
print(model_top10.summary())




VIF 점검 결과:
                  feature       VIF
0                   const  1.000000
1                  POST_c  1.878624
2   DOGE_Intensity_POST_c  1.878624
3      Population_Density  6.836624
4  College_Educated_Share  8.018357
5            Industry_PC1  7.681492
6            Industry_PC2  6.172307

OLS 회귀 결과:
                            OLS Regression Results                            
Dep. Variable:       Continued Claims   R-squared:                       0.164
Model:                            OLS   Adj. R-squared:                  0.161
Method:                 Least Squares   F-statistic:                     6.865
Date:                Sat, 10 May 2025   Prob (F-statistic):            0.00570
Time:                        21:06:43   Log-Likelihood:                -17120.
No. Observations:                1720   AIC:                         3.425e+04
Df Residuals:                    1713   BIC:                         3.429e+04
Df Model:                           6                   

C:\Users\mikey\AppData\Local\Temp\ipykernel_21724\20729387.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_top10[to_scale] = scaler.fit_transform(df_top10[to_scale])


In [13]:
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from statsmodels.stats.outliers_influence import variance_inflation_factor

# District of Columbia만 필터링
df_dc = df_merged[df_merged['State'] == 'District of Columbia'].copy()

# PCA 재계산
industry_cols = [col for col in df_x.columns if col not in ['State', 'Population_Density', 'College_Educated_Share']]
X_industry_dc = df_dc[industry_cols].fillna(0)
pca = PCA(n_components=2)
industry_pca_dc = pca.fit_transform(X_industry_dc)
df_dc['Industry_PC1'] = industry_pca_dc[:, 0]
df_dc['Industry_PC2'] = industry_pca_dc[:, 1]

# 상호작용항 중심화
df_dc['DOGE_Intensity_c'] = df_dc['DOGE_Intensity_WeightedAverage'] - df_dc['DOGE_Intensity_WeightedAverage'].mean()
df_dc['POST_c'] = df_dc['POST'] - df_dc['POST'].mean()
df_dc['DOGE_Intensity_POST_c'] = df_dc['DOGE_Intensity_c'] * df_dc['POST_c']

# 표준화
scaler = StandardScaler()
to_scale = ['Population_Density', 'College_Educated_Share', 'Industry_PC1', 'Industry_PC2', 'DOGE_Intensity_POST_c']
df_dc.loc[:, to_scale] = scaler.fit_transform(df_dc[to_scale])

# 독립변수 설정
indep_vars = ['POST_c', 'DOGE_Intensity_POST_c', 'Population_Density',
              'College_Educated_Share', 'Industry_PC1', 'Industry_PC2']

# 표준편차 거의 0인 변수 제거
stds = df_dc[indep_vars].std()
low_var_cols = stds[stds < 1e-8].index.tolist()
print(f"표준편차 ≈0 제거 변수: {low_var_cols}")
valid_vars = [v for v in indep_vars if v not in low_var_cols]

# VIF 계산
X_check = sm.add_constant(df_dc[valid_vars])
vif_data = pd.DataFrame()
vif_data['feature'] = X_check.columns
vif_data['VIF'] = [variance_inflation_factor(X_check.values, i) for i in range(X_check.shape[1])]
print("\nVIF 결과:")
print(vif_data)

# VIF >10인 변수 제거
high_vif_cols = vif_data[vif_data['VIF'] > 10]['feature'].tolist()
high_vif_cols = [col for col in high_vif_cols if col != 'const']
print(f"\nVIF>10 제거 변수: {high_vif_cols}")
final_vars = [v for v in valid_vars if v not in high_vif_cols]

# 최종 OLS 실행
X = sm.add_constant(df_dc[final_vars])
y = df_dc['Continued Claims']
full_data = pd.concat([X, y], axis=1).dropna()
X = full_data[X.columns]
y = full_data['Continued Claims']

if len(full_data) > len(final_vars) + 1:
    model_dc = sm.OLS(y, X).fit(cov_type='HC3')
    print("\nOLS 결과:")
    print(model_dc.summary())
else:
    print("\n데이터가 부족하여 회귀분석을 실행할 수 없습니다.")


표준편차 ≈0 제거 변수: ['DOGE_Intensity_POST_c', 'Population_Density', 'College_Educated_Share', 'Industry_PC1']

VIF 결과:
        feature       VIF
0         const  1.000000
1        POST_c  1.000571
2  Industry_PC2  1.000571

VIF>10 제거 변수: []

OLS 결과:
                            OLS Regression Results                            
Dep. Variable:       Continued Claims   R-squared:                       0.463
Model:                            OLS   Adj. R-squared:                  0.456
Method:                 Least Squares   F-statistic:                     52.79
Date:                Sat, 10 May 2025   Prob (F-statistic):           1.54e-18
Time:                        21:08:14   Log-Likelihood:                -1477.5
No. Observations:                 172   AIC:                             2961.
Df Residuals:                     169   BIC:                             2970.
Df Model:                           2                                         
Covariance Type:                  HC3       